# 📘 Notebook 11 · 大模型 + 另类数据

> 2023-2025 年量化最大的变化是 LLM。**头部私募从"用 BERT 做 NLP"升级到"用 GPT/Claude 做 alpha 研究 + 信号生成"**。

**学完你会：**

- 用 GPT-4 / Claude API 做新闻情绪分析、研报摘要、事件抽取
- 知道金融领域 LLM 的现状（FinGPT, BloombergGPT, DeepSeek-Finance）
- 看清"另类数据"赛道：卫星、信用卡、Web scraping、社交
- 知道头部私募怎么用 LLM 做 alpha 研究自动化
- 避开 LLM 在量化里最常见的三个坑

**预计 6-8 小时**

**前置**：Notebook 09。会用任意 LLM API（OpenAI / Anthropic / 国内开源模型）

## 1. LLM 在量化里能做什么 / 不能做什么

### 1.1 能做的（已被验证）

| 任务 | 例子 | 价值 |
|---|---|---|
| **文本情绪** | "茅台业绩不及预期" → 利空 | 短期事件交易 |
| **结构化抽取** | 公告 → {事件类型, 金额, 时间} | 替代人工标注 |
| **研报摘要** | 100 页研报 → 5 个要点 | 研究员效率 |
| **代码生成** | "写一个 IC 测试函数" | 研究员工具 |
| **因子发现** | "给我 10 个动量类因子的 idea" | 因子库扩展 |
| **多模态** | 财报 PDF + 数字 + 表格 | 全方位理解 |

### 1.2 不能做的（被吹过头）

| 不擅长 | 原因 |
|---|---|
| **直接预测股价** | 噪声主导，LLM 没特殊优势 |
| **做出比 LightGBM 好的 alpha** | 数值预测不是 LLM 强项 |
| **替代经济学逻辑** | LLM 会瞎编理由，不会真证伪 |
| **跑数学密集型的回测** | 慢、贵、不准 |

### 1.3 头部私募 LLM 应用真相

**2024-2025 年观察**：

- **研究效率工具**：90% 头部私募已部署内部 LLM 助手
- **新闻情绪信号**：50% 团队在用，**单独 alpha 不强但可叠加**
- **自动因子挖掘**：实验阶段，效果不稳定
- **替代研究员**：远未到那一天

## 2. 新闻情绪分析：最经典的 LLM 应用

### 2.1 任务定义

输入：一条新闻 / 公告 / 推文
输出：对相关股票的情绪 (-1 ~ +1)，可能还有 (主题, 实体, 强度)

### 2.2 三代做法对比

| 代次 | 方法 | 效果 |
|---|---|---|
| G1 | 情感词典（VADER）| 简单，30-40% 准确 |
| G2 | FinBERT 等专用模型 | 70-80% 准确 |
| G3 | GPT-4 / Claude few-shot | 80-90% 准确 |

但是注意：**准确率高 ≠ 能赚钱**。重要的是**信号在市场上的 alpha**，而不是分类器的 accuracy。

### 2.3 用 Claude API 做新闻情绪分析

In [ ]:
# 注意:需要 API key（Anthropic / OpenAI / DeepSeek 等兼容接口都行）

DEMO_CODE = '''import os
from anthropic import Anthropic

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

PROMPT = '''你是一名金融情绪分析师。对以下新闻进行评估:

[新闻]
{news}

请只用 JSON 回复:
{{
  "stock": "涉及的股票",
  "sentiment": -1 到 1 的浮点数,
  "category": "earnings/regulatory/m&a/management/product/other",
  "confidence": 0 到 1 的浮点数,
  "reasoning": "一句话理由"
}}'''

def analyze_news(news_text):
    msg = client.messages.create(
        model="claude-opus-4-7",
        max_tokens=512,
        messages=[{"role": "user", "content": PROMPT.format(news=news_text)}],
    )
    import json, re
    text = msg.content[0].text
    text = re.sub(r"```(?:json)?\n?", "", text).strip("`").strip()
    return json.loads(text)


# 测试
news = "贵州茅台公布2025年三季报,营收同比下降3.2%,净利润下降5.8%,低于市场预期。"
print(analyze_news(news))
# 期望输出:
# {"stock": "贵州茅台", "sentiment": -0.7, "category": "earnings",
#  "confidence": 0.9, "reasoning": "营收净利双降且低于预期"}'''
print(DEMO_CODE)
print('\n上面代码需要 API key + 联网。本机先 pip install anthropic 或 openai。')

### 2.4 三个 Prompt 设计要点

**1. 强制结构化输出（JSON）**

不要让 LLM "自由发挥"——你拿不到的结构化数据没用。永远 JSON 输出。

**2. 给少量示例（Few-Shot）**

```
示例 1：[新闻] xxx [输出] xxx
示例 2：[新闻] xxx [输出] xxx

现在轮到你：
[新闻] {target} [输出] ?
```

复杂任务 few-shot 比 zero-shot 准确率高 10-20%。

**3. 多步推理（Chain-of-Thought）**

```
请按步骤分析：
1. 这条新闻涉及哪家公司？
2. 这是利好还是利空？为什么？
3. 影响是短期（< 1 周）、中期（1-3 月）还是长期？
4. 最后用 JSON 输出。
```

CoT 在复杂任务上能再提升 5-15%。

## 3. 把情绪分数变成 alpha 因子

得到每条新闻的情绪分数后，怎么组合成可交易的信号？

### 3.1 构造方法

```python
# 每只股票的"情绪因子"：
sentiment_score[stock, day] = 
    weighted_avg([news_sentiment for news where stock in news, date == day])

# 权重可以是：
# - 来源权重（路透 > 个股吧）
# - 时间衰减（近期权重高）
# - 信息含量（新增信息 > 老调重弹）
```

### 3.2 评估方法

和普通因子一样：
- 算 IC / Rank IC
- 分层回测
- 看与已有因子（动量、波动率）的相关性

### 3.3 真实 paper 中的结果

- Tetlock (2007) JF "Giving Content to Investor Sentiment"：情绪因子 IC ~0.02
- Heston-Sinha (2017): 媒体情绪能预测 1-2 日收益
- Ke, Kelly, Xiu (2019): 用 ML 提取的 sentiment，预测能力延伸到月度

**典型工业实战**：情绪因子单独 IC = 0.01-0.03，**与价量因子相关性低（< 0.1）**，作为多因子的补充非常合适。

### 3.4 注意事项

- **新闻发布时间 vs 价格反映时间**：必须用"新闻发布时刻"而不是"新闻日期"，否则有未来函数
- **同样的新闻被多家媒体转载**：去重很重要
- **公告 vs 评论**：公告本身的事实是 alpha，分析师的评论是 noise

## 4. 结构化信息抽取：从公告到数据库

很多 alpha 信号来自**精细的结构化数据**。例如：
- 大股东减持（多大规模？什么价位？预计完成时间？）
- 并购重组（标的是谁？金额？支付方式？）
- 高管变动（谁离开？谁接任？背景？）

传统做法：人工读公告标注。**LLM 让这件事可以自动化**。

### 4.1 Prompt 模板

In [ ]:
# Prompt 模板示例
SHAREHOLDER_REDUCTION_PROMPT = '''
你是一名金融数据分析师。请从以下公告抽取股东减持相关信息:

[公告原文]
{announcement}

请只输出 JSON:
{{
  "stock_code": "股票代码",
  "stock_name": "股票简称",
  "reducer_name": "减持人姓名/机构",
  "reducer_type": "controlling/major/management/other",
  "current_shareholding_pct": 减持前持股比例(小数),
  "reduction_method": "block_trade/centralized_bidding/agreement_transfer/other",
  "reduction_pct_max": 计划减持上限(小数),
  "reduction_period_start": "YYYY-MM-DD",
  "reduction_period_end": "YYYY-MM-DD",
  "estimated_amount_rmb": 预估金额(人民币),
  "reasoning": "一句话理由"
}}

如果某字段无法判定,填 null。
'''

# 模拟一个公告
sample_announcement = '''
贵州茅台酒股份有限公司关于持股5%以上股东减持股份预披露公告

本公司股东习酒集团(持有公司股份5.66%)拟通过集中竞价交易方式
减持本公司股份不超过12,553,400股(占公司总股本的1%)。

减持期间: 自本公告披露之日起15个交易日后的90个自然日内
(即 2025-11-15 至 2026-02-12)。

减持原因: 自身资金需求。
'''

# 用 LLM 抽取(模拟结果,真实需 API)
mock_result = {
    "stock_code": "600519",
    "stock_name": "贵州茅台",
    "reducer_name": "习酒集团",
    "reducer_type": "major",
    "current_shareholding_pct": 0.0566,
    "reduction_method": "centralized_bidding",
    "reduction_pct_max": 0.01,
    "reduction_period_start": "2025-11-15",
    "reduction_period_end": "2026-02-12",
    "estimated_amount_rmb": 12553400 * 1500,    # 假设股价 1500
    "reasoning": "5%以上大股东自身资金需求减持"
}

import json
print('抽取出的结构化数据:')
print(json.dumps(mock_result, indent=2, ensure_ascii=False))

### 4.2 这些数据能做什么策略

**策略 1：大股东减持事件研究**
- 信号：减持公告发布日
- 持仓：减持窗口内做空（如果跌幅显著）
- 学术依据：Lakonishok & Lee (2001) JFE 内部人交易

**策略 2：聚合所有结构化事件构建事件因子库**
- 增持、减持、回购、定增、并购、监管处罚...
- 每个事件单独 alpha 弱，但 50+ 事件因子组合起来 ICIR 可达 0.6+

**策略 3：监管处罚事件**
- 证监会立案调查 → 股价下跌
- 罚单金额、性质（财务造假 / 内幕交易）影响幅度

### 4.3 工业实践提醒

- **抽取错误率 5-10% 是正常的**，需要随机抽样人工 review
- **重要字段加 sanity check**：金额不超过总市值、日期合理性
- **prompt 要版本化**：换了 prompt 等于换了一份数据，必须重新跑历史
- **成本控制**：GPT-4 处理一年公告（~30 万条）成本约 $1000-3000

## 5. 用 LLM 做因子挖掘（AlphaGPT 类）

### 5.1 传统因子挖掘

```
研究员：脑子里想一个因子 → 写代码 → 跑 IC → 通过 / 失败
```

**问题**：依赖研究员经验，迭代慢，容易卡在熟悉的因子范式。

### 5.2 AlphaGPT 类做法（2023-2024 论文）

```
LLM：给一个因子假说 → 
  自动写 Python 代码实现 → 
  自动测试 → 
  根据反馈生成下一个因子
```

代表论文：
- "Alpha-GPT: Human-AI Interactive Alpha Mining"（2023）
- "FinAgent: A Multimodal Foundation Agent for Financial Trading"（2024）
- 国内华泰、华西等已发布类似框架

### 5.3 实际效果

**好的方面：**
- 一晚上能生成几百个候选因子
- LLM 见过很多 paper，能想出研究员没想过的组合
- 自动写代码 + 测试 → 大幅减少 boilerplate

**坑：**
- LLM 会"瞎编"——给出公式错误的因子
- 多数因子很快被发现已经存在或无效
- 严重的 multiple testing 问题——试 1000 个里有 50 个"显著"是必然的

### 5.4 一个最简的 AlphaGPT prompt

In [ ]:
ALPHA_PROMPT = '''
你是量化研究员。请基于"投资者过度反应"理论，生成 1 个原创因子假说。

要求：
1. 因子定义：用 Python 函数实现，输入 price (DataFrame, date × stock)、volume (DataFrame, date × stock)、market_cap (DataFrame, date × stock)，输出因子值 (DataFrame, date × stock)
2. 经济学逻辑：解释为什么这个因子能预测未来收益
3. 预期方向：因子值高的股票，未来收益是 + 还是 -？
4. 潜在风险：在什么市场环境下可能失效

只输出 JSON：
{
  "factor_name": "...",
  "formula": "Python 代码字符串",
  "rationale": "经济学逻辑",
  "expected_direction": "+/-",
  "risks": "失效场景"
}
'''

# LLM 可能的响应（模拟）
mock_alpha = {
    "factor_name": "overreaction_reversal",
    "formula": '''def factor(price, volume, market_cap):
    # 过去 5 日累计极端收益（绝对值 top 20%）后的反转因子
    ret_5d = price.pct_change(5)
    vol_ratio = volume.rolling(5).mean() / volume.rolling(60).mean()
    extreme = (ret_5d.abs() > ret_5d.abs().quantile(0.8, axis=1).values[:, None])
    # 大幅波动 + 放量 = 过度反应 → 反转
    overreaction = -ret_5d * extreme * vol_ratio
    return overreaction.shift(1)''',
    "rationale": "短期极端收益伴随放量，往往是过度反应。投资者情绪驱动的过度买入/卖出，价格偏离基本面后回归。",
    "expected_direction": "+",
    "risks": "强趋势市场下反转失效；放量也可能是真实信息冲击导致的合理价格调整"
}

import json
print('LLM 生成的候选因子：')
print(json.dumps(mock_alpha, indent=2, ensure_ascii=False))

### 5.5 完整 AlphaGPT 系统的设计

```
┌─────────────────────────────────────────────┐
│              Alpha Brain (LLM)              │
│  - 读 paper / 看历史因子                    │
│  - 提出新假说                                │
│  - 写 Python 代码                            │
└──────────────────────┬──────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────┐
│           Auto Tester (Python)              │
│  - 跑回测：IC / Rank IC / Sharpe            │
│  - 评估稳定性、与已有因子相关性             │
└──────────────────────┬──────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────┐
│          Feedback Loop (LLM)                │
│  - 分析失败原因                              │
│  - 改进假说 / 修代码                         │
│  - 进入下一轮                                │
└─────────────────────────────────────────────┘
```

**头部私募已部分实现这个系统**。但**研究员仍然不可替代**——最终的策略上线、组合优化、风险管理都需要人。

### 5.6 你能做的小项目

1. 写一个 prompt，让 LLM 生成 50 个候选因子
2. 用 Notebook 01 的 pipeline 自动测试
3. 看通过 IC > 0.03、ICIR > 0.3 的有几个
4. 在那几个里再用经济学逻辑筛
5. 把整个过程写成"AlphaGPT lite"放 GitHub

## 6. 另类数据：传统量化的下一片蓝海

### 6.1 什么是另类数据

不是来自交易所的传统价量 / 基本面数据。包括：

| 类型 | 例子 | 用途 |
|---|---|---|
| **卫星图像** | 油罐影、停车场车数 | 预测库存、零售业绩 |
| **信用卡数据** | 各品牌消费聚合 | 提前预测季度营收 |
| **网络爬虫** | 招聘信息、产品价格 | 公司经营信号 |
| **App 数据** | 下载量、DAU | 互联网公司业绩 |
| **物流数据** | 海运、快递量 | 经济活动景气 |
| **社交媒体** | Twitter / 微博情绪 | 散户情绪反指 |
| **专利数据** | 申请数、引用 | 创新能力 |
| **天气数据** | 极端气候 | 商品价格、保险 |
| **声呐 / 雷达** | 港口船只数 | 大宗商品物流 |

### 6.2 顶级另类数据供应商

| 公司 | 数据 | 价格 |
|---|---|---|
| Quandl (Nasdaq) | 综合 | $10k-100k/年/数据集 |
| Eagle Alpha | 信用卡 / 卫星 / Web | $100k+/年 |
| Yipit Data | 信用卡聚合 | $50k+/年 |
| Bloomberg ALT | 综合 | $50k+/年 |
| Orbital Insight | 卫星图像 | $100k+/年 |

→ 中型私募负担不起。所以这是**头部私募的护城河**。

### 6.3 卫星图像案例

**Walmart 财报预测**：

1. 买 1000+ Walmart 门店一个季度的卫星图像
2. 用 CV 模型数停车场车辆
3. 估算客流量趋势
4. 提前 2-3 周预测季度销售额

**精度**：±2-3% 的销售额预测，远比卖方分析师准。

历史上 RenTech、Two Sigma、Citadel 都做过类似数据，**单数据集年化贡献几亿美金 PnL**。

### 6.4 LLM + 另类数据

LLM 真正颠覆另类数据的地方：**Web 爬虫数据的结构化**。

**例子：电商价格监控**

```
原始数据：京东、淘宝、拼多多上 1 万个 SKU 的页面 HTML
传统做法：每个网站写一个 parser，维护成本高、容易坏
LLM 做法：一个通用 prompt，让 LLM 抽取 {product, price, stock, rating, review_count}
```

成本：GPT-4 处理 1 万个页面约 $200。**这种事以前需要 5 个工程师维护 1 年。**

## 7. 头部私募 LLM 应用真实工作流

### 7.1 研究员的日常已经被改变

**2022 年的研究员**：

```
9:00  早会
9:30  写代码跑 backtest
12:00 午饭
13:00 改 bug，跑下一个 backtest
17:00 写研究笔记
```

**2025 年的研究员**：

```
9:00  早会
9:30  让 LLM 写 backtest 代码草稿
9:45  自己 review + 微调
10:00 跑 backtest（同时让 LLM 起草下一个 idea）
11:00 复盘 + LLM 自动生成研究笔记
12:00 午饭
13:00 探索 LLM 生成的 5 个新因子假说
17:00 写正式研究文档（LLM 起草）
```

**效率提升 2-3 倍**，但研究员**没失业**——任务更聚焦在"判断"和"创意"，而不是写 boilerplate 代码。

### 7.2 内部 LLM 工具栈

头部私募内部典型的 LLM 系统：

```
┌─ Code Assistant     代码助手（Copilot + 内部 fine-tune）
├─ Research Assistant 研究助手（读 paper / 写 summary）
├─ Strategy Generator 策略生成器（AlphaGPT 类）
├─ News Pipeline      新闻情绪 / 事件抽取（实时）
├─ Document Search    内部 RAG（找历史 backtest / 内部知识）
└─ Risk Monitor       异常文本检测（公告 + 监管文件）
```

### 7.3 关键现实

- 这些系统**不会让研究员"产出十倍 alpha"**
- 它们让研究员"少写十倍重复劳动"
- 真正的护城河是**数据 + 算力 + 团队**，LLM 是工具不是 alpha

## 8. 用 LLM 做量化的三大陷阱

### 8.1 陷阱 1：LLM 输出当真理

LLM 会**自信地编造**。例如问"贵州茅台 2024 年 Q3 净利润是多少"，可能给你一个看起来合理但完全错误的数字。

**对策：**
- 数值类问题：永远用 API + 数据库交叉验证
- 推理类问题：让 LLM 给出"推理过程"，自己检查
- 千万**不要直接用 LLM 输出做交易决策**

### 8.2 陷阱 2：训练数据污染（Data Contamination）

LLM 训练时已经"看过"很多金融论文、Wind 数据库的内容。

**例子**：让 LLM 预测 2023 年 1 月某只股票的走势，它可能"记得"答案。
- 这意味着你的回测看起来漂亮，实际是数据泄露
- 上线后立刻翻车

**对策：**
- 只用 LLM 处理**新发布**的文本（公告、新闻当日发布）
- 不要让 LLM "预测过去"
- 关键模型在自己的私有数据上 fine-tune，避免依赖通用 LLM

### 8.3 陷阱 3：成本失控

- GPT-4 处理 1 篇研报 ~$0.10
- 1000 只股票 × 100 篇/年 = 10 万次调用 = $10000
- 实时新闻流：每天 10000 条 = $1000/day = **$30000/月**

**对策：**
- 用便宜模型（GPT-3.5 / Claude Haiku / DeepSeek）做 first-pass 过滤
- 只对"重要的"内容用最强模型
- 缓存：同样的输入不要算两次
- 批量 API（OpenAI / Anthropic 都有 50% off batch）

## 9. 学习路径

### 9.1 入门

```
☐ 用 ChatGPT / Claude 网页版做 10 次新闻情绪分析
☐ 学会 API 调用（Anthropic / OpenAI SDK）
☐ 写一个 prompt 做"事件抽取"，处理 100 条公告
☐ 评估抽取准确率（自己人工标 50 条）
```

### 9.2 进阶

```
☐ 把情绪 / 事件信号变成因子，做 IC 测试
☐ 学习 LangChain / LlamaIndex 做 RAG
☐ 试 fine-tune 一个开源金融模型（如 FinGPT）
☐ 部署一个简单的研究 chatbot 给自己的 Jupyter 用
```

### 9.3 高阶

```
☐ 复刻一篇 AlphaGPT paper
☐ 自建另类数据 + LLM pipeline（电商、招聘、社交某一类）
☐ 多 agent 协作（一个 agent 出 idea、一个测试、一个总结）
☐ 在自家私有数据上 fine-tune 大模型
```

### 9.4 国内开源金融 LLM

| 模型 | 团队 | 特点 |
|---|---|---|
| **FinGPT** | 哥大 + 多家 | 早期金融 LLM，可商用 |
| **DISC-FinLLM** | 复旦 | 中文金融对话 |
| **CFGPT** | 中科院 | 中文金融预训练 |
| **Yi-Finance** | 零一万物 | 商用质量 |

**国内首选**：用 DeepSeek、Qwen 系列做 first-pass，用 GPT-4 / Claude 做高质量精排。

## 10. 推荐资源

**Paper：**
- Lopez-Lira & Tang (2023) "Can ChatGPT Forecast Stock Price Movements?"
- BloombergGPT 技术报告（2023）
- DeepSeek-V2、Qwen-Finance 报告
- FinGPT 系列

**实战教程：**
- LangChain 官方教程
- LlamaIndex 中文文档
- AlphaGPT GitHub（多个开源实现）

**社区：**
- AI4Finance Foundation（FinGPT、FinRL 维护方）
- WallStreetBets（看散户情绪反指）

---

**下一节：Notebook 12 · Web3 最新生态** — L2、Account Abstraction、Intent、Restaking、永续 DEX 的最新进展。

跨入 Web3 的最新前沿。